# Plenary 8 · Three customs questions, three numbers

Workshop **Data Driven Domestic Revenue Mobilization**, Bangkok, October 2026.

Each question is one query and one number: how much duty is at stake in low valuations, how much
duty relief costs and who uses more than allowed, and which importers the VAT office does not see.
Then the three are ranked by effort and yield, next to the tax side's three.

Press ▶ on each cell from top to bottom. Change only the sliders. Everything here is synthetic.

In [ ]:
#@title 0 · Where the data comes from
#@markdown Leave these as they are. Your analysts can point DATA_URL at their own extract (section 6).
DATA_URL = "https://raw.githubusercontent.com/FrancoisChastel/CustomsWorkshopNotebooks/data-v2/data"  #@param {type:"string"}
INSTALL = True  #@param {type:"boolean"}

In [ ]:
#@title 0 · Setup (run me first)
import sys, subprocess, warnings
warnings.filterwarnings("ignore")
if INSTALL and "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/FrancoisChastel/CustomsWorkshopNotebooks@data-v2"], check=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from workshop_methods import notebook as nb
from workshop_methods.data import load_json, load_tables
from workshop_methods import quick_queries as qq
nb.use_style()
pd.set_option("display.max_colwidth", 40)
T = load_tables(DATA_URL)
print(f"Loaded {len(T)} tables, {len(T['declarations']):,} declaration lines. Ready.")

In [ ]:
#@title 0 · Prepare the tables
d, tariff, traders = T["declarations"], T["tariff"], T["traders"]
name = traders.set_index("trader_ref")["trader_id"]
name = name.where(name != "", traders.set_index("trader_ref").index.to_series())
months = sorted(d["month"].unique())
this_quarter = months[-3:]
print(f"Two years of declarations, {months[0]} to {months[-1]}; this quarter is {this_quarter[0]} to {this_quarter[-1]}.")

## 1 · The loop in one table

In [ ]:
#@title 1 · Extract, size, rank, act, measure
loop = pd.DataFrame({
    "valuation": ["declarations and tariff", "duty at stake", "importers by duty at stake",
                  "valuation query, post-clearance audit", "query raised, upheld, duty recovered"],
    "duty relief": ["declarations under relief, licences", "duty foregone", "beneficiaries above their limit",
                    "licence review", "licence suspended, duty recovered"],
    "import VAT": ["declarations, tax registry, VAT returns", "imports of unregistered importers", "largest first",
                   "referral to VAT registration or audit", "registered, VAT assessed"],
}, index=["extract: the data you pull", "size: one number", "rank: who first", "act: what an officer does",
          "measure: the outcome field it creates"])
nb.show(loop)
nb.finding("Each question needs one extract and creates one outcome field; without the outcome field, the next "
           "ranking cannot learn which queries paid.")

## 2 · Valuation leakage

In [ ]:
#@title 2 · Duty at stake in low valuations
THRESHOLD = 0.70  #@param {type:"slider", min:0.5, max:0.9, step:0.05}
nb.steps("How much duty would we collect if low-valued lines were declared at the typical value?",
         "Drop lines whose weight and quantity disagree (a units error is not a price).",
         f"For each goods code and origin, compare value per kilo with the typical value; keep lines below {THRESHOLD:.0%} of it.",
         "Duty at stake = (typical − declared value per kilo) × weight × duty rate, summed.")
stake = qq.duty_at_stake(d, tariff, THRESHOLD)
quarter = stake[stake["month"].isin(this_quarter)]
nb.finding(f"Duty at stake this quarter: LCU {quarter['duty_at_stake'].sum():,.0f}, on {len(quarter)} lines from "
           f"{quarter['trader_ref'].nunique()} importers (two years: LCU {stake['duty_at_stake'].sum():,.0f}).")

In [ ]:
#@title 2 · Who carries it: the top importers
top = stake.groupby("trader_ref").agg(duty_at_stake=("duty_at_stake", "sum"), lines=("duty_at_stake", "size"),
                                      median_ratio=("ratio_to_peers", "median")).nlargest(20, "duty_at_stake")
top.index = top.index.map(name)
nb.show(top, formats={"duty_at_stake": "{:,.0f}", "median_ratio": "{:.2f}"})
share = top["duty_at_stake"].head(3).sum() / stake["duty_at_stake"].sum()
nb.finding(f"Importers {', '.join(top.index[:3])} carry the most duty at stake ({share:.0%} of the total between them, "
           f"on {top['lines'].head(3).sum()} lines): a short list for valuation queries.")

In [ ]:
#@title 2 · By chapter
by_chapter = stake.assign(chapter=stake["ahtn8"].str[:2]).groupby("chapter")["duty_at_stake"].sum().sort_values()
fig, ax = nb.chart(f"Chapter {by_chapter.index[-1]} holds the largest share of duty at stake")
colors = [nb.COLORS["amber"] if c == by_chapter.index[-1] else nb.COLORS["accent"] for c in by_chapter.index]
ax.barh(by_chapter.index, by_chapter.values / 1e6, color=colors)
ax.set_xlabel("duty at stake, LCU million"); plt.show()
nb.finding(f"Chapter {by_chapter.index[-1]} holds {by_chapter.iloc[-1] / by_chapter.sum():.0%} of the duty at stake.")

In [ ]:
#@title 2 · The outcome to record
record = pd.DataFrame({"importer": top.index[:5], "query raised": "", "upheld": "", "duty recovered (LCU)": ""})
nb.show(record.set_index("importer"))
nb.finding("Record one row per importer acted on: next quarter's ranking learns from these four columns.")

## 3 · Exemptions and duty relief

In [ ]:
#@title 3 · Duty foregone by scheme
nb.steps("What does duty relief cost, and under which scheme?",
         "Take import lines entered under a relief scheme.", "Duty foregone = value × the normal (MFN) duty rate.")
foregone = qq.foregone_duty(d, tariff)
fig, ax = nb.chart(f"{foregone.index[0]} accounts for the most duty foregone")
ax.barh(foregone.index[::-1], foregone.values[::-1] / 1e6, color=nb.COLORS["accent"])
ax.set_xlabel("duty foregone over two years, LCU million"); plt.show()
nb.finding(f"Relief costs LCU {foregone.sum():,.0f} in duty over two years; the largest scheme is {foregone.index[0]} "
           f"({foregone.iloc[0] / foregone.sum():.0%}).")

In [ ]:
#@title 3 · Beneficiaries above their licence limit
nb.steps("Does anyone use relief beyond what its licence allows?",
         "Sum each beneficiary's relieved value over the last 12 months.", "Divide by its licence limit.")
use = qq.relief_overuse(d, T["licences"])
view = use.head(10).set_index("beneficiary_ref")[["scheme", "relief_value_12m", "licence_limit_lcu", "use_of_limit"]]
view.index = view.index.map(name)
nb.show(view, formats={"relief_value_12m": "{:,.0f}", "licence_limit_lcu": "{:,.0f}", "use_of_limit": "{:.2f}"},
        highlight=view["use_of_limit"] > 1)
over = view[view["use_of_limit"] > 1]
nb.finding(f"{len(over)} beneficiaries used more than their licence allows ({', '.join(over.index)}, at "
           f"{over['use_of_limit'].min():.2f} to {over['use_of_limit'].max():.2f} times the limit).")

## 4 · Import VAT

In [ ]:
#@title 4 · Large importers the VAT office does not see
VAT_THRESHOLD = 1_500_000  #@param {type:"integer"}
nb.steps("Which importers above the VAT threshold are not registered for VAT?",
         "Sum each importer's imports over the last 12 months.",
         "Look up its tax ID in the tax registry; keep those not found or not registered for VAT.")
list1, list2 = qq.import_vat_lists(d, traders, T["tax_registry"], T["vat_returns"], VAT_THRESHOLD)
view = list1.head(10)[["customs_tin", "import_value_12m", "import_vat_paid"]]
view.index = view.index.map(name)
nb.show(view, formats={"import_value_12m": "{:,.0f}", "import_vat_paid": "{:,.0f}"})
nb.finding(f"{len(list1)} importers above the threshold are not registered for VAT or not found in the registry; "
           f"they imported LCU {list1['import_value_12m'].sum():,.0f} in 12 months.")

In [ ]:
#@title 4 · Import VAT claimed against import VAT paid
nb.steps("Does anyone claim more import VAT on its returns than it paid at the border?",
         "Sum import VAT paid at the border and import VAT credit claimed on returns, 12 months.",
         "Largest difference first.")
view = list2.head(8)[["import_vat_paid", "import_vat_credit", "credit_minus_paid", "credit_to_paid"]]
view.index = view.index.map(name)
nb.show(view, formats={"import_vat_paid": "{:,.0f}", "import_vat_credit": "{:,.0f}", "credit_minus_paid": "{:,.0f}", "credit_to_paid": "{:.1f}x"})
nb.finding(f"{view.index[0]} and {view.index[1]} claim {view['credit_to_paid'].iloc[0]:.1f} and {view['credit_to_paid'].iloc[1]:.1f} times "
           "the import VAT they paid at the border: a reconciliation customs and tax can only do together.")

## 5 · Rank by effort and yield

In [ ]:
#@title 5 · Effort against yield, customs and tax side by side
#@markdown Hours of analyst and officer time for each question; the tax facilitator edits the tax side's three.
HOURS_VALUATION = 40  #@param {type:"slider", min:5, max:200, step:5}
HOURS_RELIEF = 20  #@param {type:"slider", min:5, max:200, step:5}
HOURS_IMPORT_VAT = 60  #@param {type:"slider", min:5, max:200, step:5}
TAX_1 = "Non-filers"  #@param {type:"string"}
TAX_1_YIELD_M = 20  #@param {type:"number"}
TAX_1_HOURS = 50  #@param {type:"slider", min:5, max:200, step:5}
TAX_2 = "Withholding gaps"  #@param {type:"string"}
TAX_2_YIELD_M = 12  #@param {type:"number"}
TAX_2_HOURS = 30  #@param {type:"slider", min:5, max:200, step:5}
TAX_3 = "Refund claims"  #@param {type:"string"}
TAX_3_YIELD_M = 30  #@param {type:"number"}
TAX_3_HOURS = 120  #@param {type:"slider", min:5, max:200, step:5}
VAT_RATE = 0.10  #@param {type:"number"}
MARKUP = 1.25  #@param {type:"number"}
#@markdown Yields a year: valuation = duty at stake; relief = duty on relief beyond licence limits;
#@markdown import VAT = VAT on the value the unregistered importers add (imports × (mark-up − 1) × VAT rate).
customs = pd.DataFrame({
    "yield_lcu_m": [stake["duty_at_stake"].sum() / 2 / 1e6, qq.relief_excess_duty(d, tariff, T["licences"]) / 1e6,
                    list1["import_value_12m"].sum() * (MARKUP - 1) * VAT_RATE / 1e6],
    "hours": [HOURS_VALUATION, HOURS_RELIEF, HOURS_IMPORT_VAT], "side": "customs",
}, index=["valuation", "duty relief", "import VAT"])
tax = pd.DataFrame({"yield_lcu_m": [TAX_1_YIELD_M, TAX_2_YIELD_M, TAX_3_YIELD_M], "hours": [TAX_1_HOURS, TAX_2_HOURS, TAX_3_HOURS],
                    "side": "tax"}, index=[TAX_1, TAX_2, TAX_3])
grid = pd.concat([customs, tax])
grid["yield_per_hour"] = grid["yield_lcu_m"] / grid["hours"]
fig, ax = nb.chart(f"Best yield for the effort: {grid['yield_per_hour'].idxmax()}")
for side, color in [("customs", nb.COLORS["accent"]), ("tax", nb.COLORS["amber"])]:
    part = grid[grid["side"] == side]
    ax.scatter(part["hours"], part["yield_lcu_m"], s=90, color=color, label=side)
    for label, row in part.iterrows():
        ax.annotate(label, (row["hours"], row["yield_lcu_m"]), textcoords="offset points", xytext=(6, 4), fontsize=9)
ax.axvline(grid["hours"].median(), color=nb.COLORS["border"]); ax.axhline(grid["yield_lcu_m"].median(), color=nb.COLORS["border"])
ax.set_xlabel("effort, hours"); ax.set_ylabel("yield a year, LCU million"); ax.legend(); plt.show()
nb.finding(f"At these estimates {grid['yield_per_hour'].idxmax()} gives the most per hour "
           f"(LCU {grid['yield_per_hour'].max():.2f} million an hour); move the sliders to your own effort figures.")

## 6 · For your analysts

The three queries in pandas and in SQL, side by side, against the same CSV files. The SQL runs in
DuckDB here and pastes into most warehouses once the column names are mapped to yours.

In [ ]:
#@title 6 · The three queries in SQL (DuckDB)
import duckdb
con = duckdb.connect()
for table in ["declarations", "tariff", "licences", "tax_registry", "vat_returns", "traders"]:
    con.register(table, T[table])
relief_sql = """
    SELECT d.relief_scheme, ROUND(SUM(d.customs_value_lcu * t.duty_mfn / 100)) AS foregone
    FROM declarations d JOIN tariff t USING (ahtn8)
    WHERE d.relief_scheme <> '' AND d.flow = 'import'
    GROUP BY 1 ORDER BY 2 DESC"""
vat_sql = f"""
    WITH imports AS (
        SELECT trader_ref, SUM(customs_value_lcu) AS import_value_12m
        FROM declarations WHERE flow = 'import' AND procedure = 'IM4' AND month > '{months[-13]}'
        GROUP BY 1)
    SELECT i.trader_ref, tr.trader_id, i.import_value_12m
    FROM imports i JOIN traders tr USING (trader_ref)
    LEFT JOIN tax_registry r ON r.tin = tr.trader_id
    WHERE i.import_value_12m > {VAT_THRESHOLD} AND (r.tin IS NULL OR NOT r.vat_registered)
    ORDER BY 3 DESC"""
sql_relief = con.execute(relief_sql).df().set_index("relief_scheme")["foregone"]
sql_vat = con.execute(vat_sql).df()
pandas_relief = qq.foregone_duty(d, tariff).reindex(sql_relief.index)
same = np.allclose(sql_relief, pandas_relief, rtol=1e-6) and set(sql_vat["trader_ref"]) == set(list1.index)
print(relief_sql); print(vat_sql)
nb.finding("The SQL and the pandas versions give the same numbers." if same else "The SQL and pandas versions differ: check the column map.")

**Valuation in SQL.** The typical value per kilo is a median per goods code and origin over 12 months
(`MEDIAN(customs_value_lcu / net_weight_kg) GROUP BY ahtn8, origin_country`); join it back to each line,
keep the lines below 70% of it, and sum `(median − declared) × net_weight_kg × duty_rate / 100`. In
Python it is `workshop_methods.quick_queries.duty_at_stake`, which also drops lines that fail the units
check first.